In [ ]:
#Imports
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
import sys
import os
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath("../src"))
sys.path.append(os.path.abspath("../data"))

import preprocessing as prep
import utils as utls
import data_splitting as data


In [ ]:
df_raw = pd.read_csv("../data/raw/rendimiento_estudiantes_dev.csv") 
#print(df_raw.sample(15))


## Analisis exploratorio de datos

#### 1.1 Graficos y visualizacion de datos

Histogramas

In [ ]:
df = df_raw.copy()

ord_rend = {
    "Insuficiente": 0,
    "Regular": 1,
    "Bueno": 2, 
    "Excelente": 3
}

ord_sem = {
    "2022-1": 0,
    "2022-2": 1,
    "2023-1": 2,
    "2023-2": 3,
    "2024-1": 4,
    "2024-2": 5
}

df = prep.ordinal_encoding(df, ord_rend, "rendimiento")
df = prep.ordinal_encoding(df, ord_sem, "semestre") # Me importa el orden cronologico, se podrian hacer solo dos columnas de anio y semestre

fig, axes = plt.subplots(5, 3, figsize=(16, 14))
axes = axes.flatten()

cols = df.columns.tolist()  # o listá las columnas manualmente

for i, col in enumerate(cols):
    axes[i].hist(df[col].dropna(), bins=30, color='steelblue', edgecolor='white')
    axes[i].set_title(col, fontsize=13)
    axes[i].set_xlabel('')
    axes[i].tick_params(labelsize=10)

# Ocultar subplots vacíos si sobran
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribución de variables', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

Se puede ver que en escuelas algunos estan en minuscula

Boxplots


In [ ]:
feats = ["horas_estudio","asistencia","nota_previa","horas_sueno","participacion","horas_extracurricular", "distancia_escuela_km","nivel_socioeconomico","tamano_clase"]
utls.boxplots(df, feats)

Mucha cantidad de outliers evidentes en nota previa sugiere que hay notas asignadas de 0 a 100
Muchos outliers en horas extracurricular y distancia escuela

#### Heatmap de correlaciones

In [ ]:
f = feats + ["rendimiento"]

plt.figure(figsize=(10, 8))
sns.heatmap(df[f].corr(), annot=True, fmt='.2f', 
            cmap='coolwarm', center=0)
plt.title("Correlaciones entre features")
plt.tight_layout()
plt.show()

In [ ]:
df['rendimiento_bin'] = (df['rendimiento'] != 0).astype(int)


fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=130)

# Multiclase
sns.countplot(data=df, x='rendimiento', ax=axes[0], color='steelblue')
axes[0].set_title('Distribución multiclase', fontsize=13, fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Cantidad')
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{int(bar.get_height())}', ha='center', fontsize=10)

# Binaria
sns.countplot(data=df, x='rendimiento_bin', ax=axes[1], color='steelblue')
axes[1].set_title('Distribución binaria', fontsize=13, fontweight='bold')
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['Desaprobado (0)', 'Aprobado (1)'])
axes[1].set_xlabel('')
axes[1].set_ylabel('Cantidad')
for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 f'{int(bar.get_height())}', ha='center', fontsize=10)

plt.suptitle('Distribución de la variable objetivo', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
df = prep.one_hot_encoding(df, "escuela")
df = prep.notes_scale(df)
df = prep.school_letters(df)

#### 1.2 Division por escuelas

In [ ]:
feats_numericas = ['horas_estudio', 'asistencia', 'nota_previa', 'horas_sueno',
                   'participacion', 'horas_extracurricular', 'distancia_escuela_km',
                   'nivel_socioeconomico', 'tamano_clase']

label_rend = {v: k for k, v in ord_rend.items()}
label_sem  = {v: k for k, v in ord_sem.items()}


utls.plot_school_analysis(df, feats_numericas, label_rend, label_sem)

En el boxplot de nivel socioeconomico se puede ver que las medianas y rangos de cada escuela son muy diferentes, contra horas de sueno por ejemplo que tiene una distribucion casi identica para todas las escuelas. 

Las mas diferentes son asistencia, nivel socioeconomico, nota previa, tamano clase.

se puede pensar que esto afecta directamente el rendimiento, en D y G el rendimiento es mucho menor que que en C que es casi todo excelente. Se podria inferir que esas features que son las que mas varian por escuela son las razones por las diferencias de rendimientos. 

casi que no hay variacion entre semestres, pero si enormemente entre escuelas.

En cuanto a las correlaciones, se puede ver que si bien los valores absolutos difieren, en todas las escuelas las features con mas correlaciones mas altas con rendimiento (target) son las mismas: horas estudio, nota previa, asistencia, y mas baja pero igual participacion.

#### 1.3 Limpieza e imputacion de datos


Teniendo en cuenta que las distribuciones de ciertas features varian mucho segun escuela, la limpieza de datos como la imputacion de valores faltantes se va a realizar segun escuela. 

In [ ]:
#Antes de imputar hago el split de train y test

train_set, val_set = data.train_val_split(df)

